# GSE42955: source validation, sample manifest, quality control, and limma

This notebook documents source and sample validation, QC, and the approved DCM-versus-control limma analysis for GSE42955. GEO-labeled ischemic samples are excluded; all 17 labeled DCM and non-failing control samples are retained. The consensus analysis is a separate stage and is not performed here.

In [4]:
# Find the repository root so paths work from the notebook directory or project root.
root_candidates <- c(".", "..", "../..")
root_candidates <- root_candidates[file.exists(file.path(root_candidates, "plan/analysis_plan.md"))]
if (length(root_candidates) == 0L) stop("Could not locate the project root.")
project_root <- normalizePath(root_candidates[[1]], winslash = "/")

# Keep the normalized GEO matrix outside data/raw, which is reserved from edits.
processed_dir <- file.path(project_root, "data", "processed")
dir.create(processed_dir, recursive = TRUE, showWarnings = FALSE)
series_matrix_file <- file.path(processed_dir, "GSE42955_series_matrix.txt.gz")
source_url <- "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE42nnn/GSE42955/matrix/GSE42955_series_matrix.txt.gz"
if (!file.exists(series_matrix_file)) {
  download.file(source_url, destfile = series_matrix_file, mode = "wb", method = "libcurl")
}
stopifnot(file.exists(series_matrix_file), file.info(series_matrix_file)$size > 0)
cat("Processed GEO matrix:", source_url, "\n")
cat("Saved to:", series_matrix_file, "\n")
cat("Compressed file size:", file.info(series_matrix_file)$size, "bytes\n")

Processed GEO matrix: https://ftp.ncbi.nlm.nih.gov/geo/series/GSE42nnn/GSE42955/matrix/GSE42955_series_matrix.txt.gz 


Saved to: /home/user/biom5350/biom5350-dcm-consensus/data/processed/GSE42955_series_matrix.txt.gz 


Compressed file size: 3308360 bytes


In [5]:
# Load tidyverse/readr tools to parse GEO metadata into a one-row-per-sample table.
suppressPackageStartupMessages(library(tidyverse))
series_lines <- readr::read_lines(series_matrix_file, progress = FALSE)
metadata_keys <- c(
  "!Sample_geo_accession", "!Sample_title", "!Sample_source_name_ch1",
  "!Sample_platform_id", "!Sample_data_row_count"
)
metadata_pattern <- paste0("^(", paste(metadata_keys, collapse = "|"), ")\\t")
metadata_lines <- series_lines[stringr::str_detect(series_lines, metadata_pattern)]
if (length(metadata_lines) != length(metadata_keys)) {
  stop("Expected one GEO metadata row for each required sample field.")
}
metadata_raw <- readr::read_tsv(
  I(stringr::str_c(metadata_lines, collapse = "\n")),
  col_names = FALSE, show_col_types = FALSE, progress = FALSE
)

# Pivot the selected GEO metadata fields so each GSM has its labels and platform together.
sample_metadata <- metadata_raw |>
  tidyr::pivot_longer(
    cols = -X1, names_to = "sample_column", values_to = "value"
  ) |>
  dplyr::mutate(sample_order = readr::parse_number(sample_column)) |>
  dplyr::select(sample_order, field = X1, value) |>
  tidyr::pivot_wider(names_from = field, values_from = value) |>
  dplyr::arrange(sample_order) |>
  dplyr::rename(
    gsm = `!Sample_geo_accession`,
    original_title = `!Sample_title`,
    original_source_label = `!Sample_source_name_ch1`,
    platform_id = `!Sample_platform_id`,
    reported_feature_count = `!Sample_data_row_count`
  )

stopifnot(nrow(sample_metadata) == 29L, dplyr::n_distinct(sample_metadata$gsm) == 29L)

In [10]:
# Verify the GEO processing annotation and that the processed matrix matches its sample metadata.
processing_lines <- series_lines[stringr::str_detect(series_lines, "^!Sample_data_processing\\t")]
processing_text <- stringr::str_to_lower(stringr::str_c(processing_lines, collapse = " "))
if (!stringr::str_detect(processing_text, "rma") ||
    !stringr::str_detect(processing_text, "log\\s*2")) {
  stop("GEO does not clearly document RMA and log2 processing; pause for review.")
}
stopifnot(identical(unique(sample_metadata$platform_id), "GPL6244"))

matrix_start <- match("!series_matrix_table_begin", series_lines)
if (is.na(matrix_start)) stop("The GEO series matrix table was not found.")
# Read only the GEO-reported number of feature rows so the closing marker is not parsed as data.
expression_table <- readr::read_tsv(
  series_matrix_file, skip = matrix_start,
  n_max = as.integer(unique(sample_metadata$reported_feature_count)),
  show_col_types = FALSE, progress = FALSE
)
matrix_gsms <- names(expression_table)[-1]
if (!setequal(sample_metadata$gsm, matrix_gsms)) {
  stop("GEO metadata GSMs do not match the processed matrix columns.")
}
if (nrow(expression_table) != 33297L || ncol(expression_table) - 1L != 29L) {
  stop("Processed matrix dimensions differ from the GEO metadata; pause for review.")
}

# The GEO processing note explicitly states log2 RMA; report the observed numeric range as a cross-check.
expression_values <- expression_table |>
  dplyr::select(-1) |>
  unlist(use.names = FALSE) |>
  as.numeric()
expression_range <- range(expression_values, na.rm = TRUE)
cat("Platform:", unique(sample_metadata$platform_id), "\n")
cat("Processed matrix dimensions:", nrow(expression_table), "features x", ncol(expression_table) - 1L, "samples\n")
cat("GEO processing annotation: RMA with log2 transformation\n")
cat("Observed expression range:", paste(signif(expression_range, 7), collapse = " to "), "\n")


Platform: GPL6244 


Processed matrix dimensions: 33297 features x 29 samples


GEO processing annotation: RMA with log2 transformation


Observed expression range: 1.80475 to 14.1064 


In [7]:
# Assign only GEO-labeled DCM and normal-heart controls; retain ICM with its exclusion reason.
sample_manifest <- sample_metadata |>
  dplyr::mutate(
    assigned_group = dplyr::case_when(
      original_source_label == "Dilated cardiomyopathy" ~ "DCM",
      original_source_label == "Normal heart" ~ "Control",
      original_source_label == "Ischemic cardiomyopathy" ~ "ICM",
      TRUE ~ "review required"
    ),
    inclusion_status = dplyr::if_else(
      assigned_group %in% c("DCM", "Control"), "included", "excluded"
    ),
    exclusion_reason = dplyr::if_else(
      assigned_group == "ICM",
      "Ischemic cardiomyopathy excluded per approved plan",
      NA_character_
    )
  ) |>
  dplyr::select(
    gsm, original_title, original_source_label, assigned_group,
    inclusion_status, exclusion_reason
  )

# Stop if a label is unrecognized or the cohort counts differ from the approved expectations.
if (any(sample_manifest$assigned_group == "review required")) {
  stop("An unrecognized GEO diagnosis label requires review before proceeding.")
}
expected_counts <- sample_manifest |>
  dplyr::count(assigned_group, inclusion_status, name = "sample_count")
print(expected_counts)
stopifnot(
  sum(sample_manifest$assigned_group == "DCM") == 12L,
  sum(sample_manifest$assigned_group == "Control") == 5L,
  sum(sample_manifest$assigned_group == "ICM") == 12L,
  sum(sample_manifest$inclusion_status == "included") == 17L
)

table_dir <- file.path(project_root, "results", "tables")
dir.create(table_dir, recursive = TRUE, showWarnings = FALSE)
manifest_file <- file.path(table_dir, "GSE42955_sample_info.csv")
readr::write_csv(sample_manifest, manifest_file)
cat("Saved sample manifest:", manifest_file, "\n")
cat("Label source: GEO !Sample_source_name_ch1; original title retained from !Sample_title.\n")

# A tibble: 3 × 3
  assigned_group inclusion_status sample_count
  <chr>          <chr>                   <int>
1 Control        included                    5
2 DCM            included                   12
3 ICM            excluded                   12


Saved sample manifest: /home/user/biom5350/biom5350-dcm-consensus/results/tables/GSE42955_sample_info.csv 


Label source: GEO !Sample_source_name_ch1; original title retained from !Sample_title.


## Stage 1 findings and comparison with GSE5406

GEO metadata yield 12 DCM, 5 normal-heart controls, and 12 ICM samples. The 17 DCM/control samples are marked included; the 12 ICM samples remain in the manifest as excluded with an explicit reason. The matrix uses GPL6244 and GEO reports RMA normalization and log2 transformation. GSE42955 provides a 29-CEL raw archive listing, unlike GSE5406, for which no raw CEL files were available in the reviewed record; no raw files were downloaded here. GSE5406 uses GPL96 and has a larger cohort (86 DCM, 16 controls, 108 ICM).

Stage 1 is complete. Stage 2 QC results are documented below. No limma model has been fit.

In [8]:
# Record package and R versions so the metadata/manifest steps are reproducible.
cat("tidyverse:", as.character(packageVersion("tidyverse")), "\n")
cat("readr:", as.character(packageVersion("readr")), "\n")
cat("\nSession information:\n")
print(utils::sessionInfo())

tidyverse: 2.0.0 


readr: 2.2.0 



Session information:


R version 4.5.2 (2025-10-31)
Platform: x86_64-pc-linux-gnu
Running under: Ubuntu 26.04 LTS

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/blas/libblas.so.3.12.1 
LAPACK: /usr/lib/x86_64-linux-gnu/lapack/liblapack.so.3.12.1;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

time zone: Etc/UTC
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] lubridate_1.9.5 forcats_1.0.1   stringr_1.6.0   dplyr_1.2.1    
 [5] purrr_1.2.2     readr_2.2.0     tidyr_1.3.2     tibble_3.3.1   
 [9] ggplot2_4.0.3   tidyverse_2.0.0

loaded via a namespace (and not attached):
 [1] bit_4.6.0          gtable_0.3.6     

# Stage 2: quality control for the 17 included samples

The processed series matrix is documented by GEO as RMA-normalized and log2-transformed. The full-matrix observed range was checked in Stage 1; this section confirms the range among the 17 DCM/control samples used for QC. The QC plots include all included samples. No ICM samples enter these calculations, no samples are removed, and no limma model is fit. For PCA, MDS, and clustering, probes must be finite across all 17 samples and zero-variance probes are omitted. Missing values are not imputed.

In [11]:
# Load plotting and data tools used for the sample-level QC summaries.
suppressPackageStartupMessages(library(tidyverse))

# Keep only the pre-approved DCM and control samples, in a stable group/GSM order.
qc_manifest <- sample_manifest |>
  dplyr::filter(inclusion_status == "included") |>
  dplyr::mutate(assigned_group = factor(assigned_group, levels = c("Control", "DCM"))) |>
  dplyr::arrange(assigned_group, gsm)
included_gsms <- qc_manifest$gsm
stopifnot(nrow(qc_manifest) == 17L, dplyr::n_distinct(included_gsms) == 17L)

# Align the expression columns to the manifest so sample labels match every QC result.
qc_expression_table <- expression_table |>
  dplyr::select(probe_id = 1, dplyr::all_of(included_gsms))
qc_expression <- qc_expression_table |>
  dplyr::select(-probe_id) |>
  as.matrix()
storage.mode(qc_expression) <- "double"
rownames(qc_expression) <- qc_expression_table$probe_id

# Count missing and non-finite values before deciding which probes support ordination.
missing_value_count <- sum(is.na(qc_expression))
nonfinite_value_count <- sum(!is.finite(qc_expression) & !is.na(qc_expression))
finite_feature <- apply(qc_expression, 1, function(x) all(is.finite(x)))
finite_expression <- qc_expression[finite_feature, , drop = FALSE]
feature_variance <- apply(finite_expression, 1, stats::var)
zero_variance_feature <- !is.finite(feature_variance) | feature_variance == 0
qc_matrix <- finite_expression[!zero_variance_feature, , drop = FALSE]

# Report the verified log2 range and QC-only feature omissions.
included_range <- range(qc_expression[is.finite(qc_expression)])
cat("Included samples:", ncol(qc_expression), "(DCM", sum(qc_manifest$assigned_group == "DCM"), "; controls", sum(qc_manifest$assigned_group == "Control"), ")\n")
cat("GEO scale annotation: RMA-normalized, log2-transformed\n")
cat("Observed range among included samples:", paste(signif(included_range, 7), collapse = " to "), "\n")
cat("Missing expression values:", missing_value_count, "\n")
cat("Non-finite, non-missing expression values:", nonfinite_value_count, "\n")
cat("Features omitted for any missing/non-finite value:", sum(!finite_feature), "of", nrow(qc_expression), "\n")
cat("Zero-variance features among finite features:", sum(zero_variance_feature), "\n")
cat("Features retained for PCA/MDS/clustering:", nrow(qc_matrix), "\n")
stopifnot(ncol(qc_matrix) == 17L, nrow(qc_matrix) > 1L)

Included samples: 17 (DCM 12 ; controls 5 )


GEO scale annotation: RMA-normalized, log2-transformed


Observed range among included samples: 1.83259 to 14.1064 


Missing expression values: 0 


Non-finite, non-missing expression values: 0 


Features omitted for any missing/non-finite value: 0 of 33297 


Zero-variance features among finite features: 0 


Features retained for PCA/MDS/clustering: 33297 


In [12]:
# Create the figure directory for the planned Stage 2 QC outputs.
figure_dir <- file.path(project_root, "results", "figures")
dir.create(figure_dir, recursive = TRUE, showWarnings = FALSE)

# Compare per-sample distributions on the verified log2 scale using finite measurements.
distribution_data <- qc_expression |>
  as.data.frame(check.names = FALSE) |>
  tibble::rownames_to_column("probe_id") |>
  tidyr::pivot_longer(-probe_id, names_to = "gsm", values_to = "expression") |>
  dplyr::filter(is.finite(expression)) |>
  dplyr::left_join(qc_manifest |> dplyr::select(gsm, assigned_group), by = "gsm")
p_distribution <- ggplot(distribution_data, aes(x = gsm, y = expression, fill = assigned_group)) +
  geom_boxplot(outlier.shape = NA, linewidth = 0.25) +
  scale_fill_manual(values = c(Control = "#4472C4", DCM = "#D55E00")) +
  labs(title = "GSE42955 expression distributions", x = "Sample (GSM)", y = "Log2 RMA expression", fill = "Group") +
  theme_minimal(base_size = 11) +
  theme(axis.text.x = element_text(angle = 60, hjust = 1), legend.position = "top")
ggplot2::ggsave(file.path(figure_dir, "GSE42955_expression_distributions.png"), p_distribution, width = 10, height = 6, dpi = 300)

# Run centered PCA without unit-variance scaling so high-variance probes keep their natural weight.
pca_fit <- stats::prcomp(t(qc_matrix), center = TRUE, scale. = FALSE)
pca_variance <- 100 * pca_fit$sdev^2 / sum(pca_fit$sdev^2)
pca_scores <- as.data.frame(pca_fit$x[, 1:2, drop = FALSE]) |>
  tibble::rownames_to_column("gsm") |>
  dplyr::left_join(qc_manifest |> dplyr::select(gsm, assigned_group), by = "gsm")
p_pca <- ggplot(pca_scores, aes(x = PC1, y = PC2, color = assigned_group, label = gsm)) +
  geom_hline(yintercept = 0, color = "grey85", linewidth = 0.3) +
  geom_vline(xintercept = 0, color = "grey85", linewidth = 0.3) +
  geom_point(size = 3) +
  geom_text(vjust = -0.8, size = 2.8, show.legend = FALSE) +
  scale_color_manual(values = c(Control = "#4472C4", DCM = "#D55E00")) +
  labs(title = "GSE42955 PCA", x = paste0("PC1 (", round(pca_variance[1], 1), "%)"), y = paste0("PC2 (", round(pca_variance[2], 1), "%)"), color = "Group") +
  theme_minimal(base_size = 11) + theme(legend.position = "top")
ggplot2::ggsave(file.path(figure_dir, "GSE42955_PCA.png"), p_pca, width = 8, height = 6, dpi = 300)

# Use classical MDS on Euclidean sample distances to show the same finite, variable probe set.
mds_fit <- stats::cmdscale(stats::dist(t(qc_matrix)), k = 2, eig = TRUE)
mds_scores <- as.data.frame(mds_fit$points) |>
  rlang::set_names(c("MDS1", "MDS2")) |>
  tibble::rownames_to_column("gsm") |>
  dplyr::left_join(qc_manifest |> dplyr::select(gsm, assigned_group), by = "gsm")
p_mds <- ggplot(mds_scores, aes(x = MDS1, y = MDS2, color = assigned_group, label = gsm)) +
  geom_hline(yintercept = 0, color = "grey85", linewidth = 0.3) +
  geom_vline(xintercept = 0, color = "grey85", linewidth = 0.3) +
  geom_point(size = 3) +
  geom_text(vjust = -0.8, size = 2.8, show.legend = FALSE) +
  scale_color_manual(values = c(Control = "#4472C4", DCM = "#D55E00")) +
  labs(title = "GSE42955 classical MDS", x = "MDS1", y = "MDS2", color = "Group") +
  theme_minimal(base_size = 11) + theme(legend.position = "top")
ggplot2::ggsave(file.path(figure_dir, "GSE42955_MDS.png"), p_mds, width = 8, height = 6, dpi = 300)

# Save a hierarchical sample dendrogram to inspect whether any sample joins the others unusually late.
sample_distance <- stats::dist(t(qc_matrix), method = "euclidean")
sample_tree <- stats::hclust(sample_distance, method = "complete")
png(file.path(figure_dir, "GSE42955_sample_clustering.png"), width = 1800, height = 1200, res = 180)
plot(sample_tree, labels = sample_tree$labels, main = "GSE42955 sample clustering", xlab = "", sub = "Euclidean distance; complete linkage; finite, non-zero-variance probes")
dev.off()

# List saved figures and summarize ordination coordinates to support visual review.
cat("Saved QC figures in:", figure_dir, "\n")
print(list.files(figure_dir, pattern = "^GSE42955_.*\\.png$", full.names = TRUE))
cat("\nPCA sample coordinates:\n")
print(pca_scores |> dplyr::arrange(PC1))
cat("\nMDS sample coordinates:\n")
print(mds_scores |> dplyr::arrange(MDS1))
cat("\nHierarchical clustering order:\n")
print(sample_tree$labels[sample_tree$order])

agg_record_17775cda5ba1 
                      2

Saved QC figures in: /home/user/biom5350/biom5350-dcm-consensus/results/figures 


[1] "/home/user/biom5350/biom5350-dcm-consensus/results/figures/GSE42955_expression_distributions.png"
[2] "/home/user/biom5350/biom5350-dcm-consensus/results/figures/GSE42955_MDS.png"                     
[3] "/home/user/biom5350/biom5350-dcm-consensus/results/figures/GSE42955_PCA.png"                     
[4] "/home/user/biom5350/biom5350-dcm-consensus/results/figures/GSE42955_sample_clustering.png"       



PCA sample coordinates:


          gsm         PC1         PC2 assigned_group
1  GSM1053915 -56.6675210 -18.7134521            DCM
2  GSM1053926 -33.3555529  18.3556719            DCM
3  GSM1053925 -17.8117740 -14.7870256            DCM
4  GSM1053937 -16.4739588  -6.1412453            DCM
5  GSM1053929 -16.4306161  15.0933703        Control
6  GSM1053933 -14.9389455 -15.4737149            DCM
7  GSM1053922  -5.3233294  19.6018110        Control
8  GSM1053938  -2.7280855  18.4676881            DCM
9  GSM1053919   0.6118655  -7.6530698            DCM
10 GSM1053924   1.9658371   3.0359065            DCM
11 GSM1053941   7.6837865  11.4949301            DCM
12 GSM1053917  18.3251276  -2.4585131            DCM
13 GSM1053935  19.0779809  -0.6272986            DCM
14 GSM1053940  23.0056659  11.8003799        Control
15 GSM1053939  23.4147375   1.2765790        Control
16 GSM1053918  31.8135844 -47.1265917            DCM
17 GSM1053942  37.8311978  13.8545744        Control



MDS sample coordinates:


          gsm        MDS1        MDS2 assigned_group
1  GSM1053915 -56.6675210 -18.7134521            DCM
2  GSM1053926 -33.3555529  18.3556719            DCM
3  GSM1053925 -17.8117740 -14.7870256            DCM
4  GSM1053937 -16.4739588  -6.1412453            DCM
5  GSM1053929 -16.4306161  15.0933703        Control
6  GSM1053933 -14.9389455 -15.4737149            DCM
7  GSM1053922  -5.3233294  19.6018110        Control
8  GSM1053938  -2.7280855  18.4676881            DCM
9  GSM1053919   0.6118655  -7.6530698            DCM
10 GSM1053924   1.9658371   3.0359065            DCM
11 GSM1053941   7.6837865  11.4949301            DCM
12 GSM1053917  18.3251276  -2.4585131            DCM
13 GSM1053935  19.0779809  -0.6272986            DCM
14 GSM1053940  23.0056659  11.8003799        Control
15 GSM1053939  23.4147375   1.2765790        Control
16 GSM1053918  31.8135844 -47.1265917            DCM
17 GSM1053942  37.8311978  13.8545744        Control



Hierarchical clustering order:


 [1] "GSM1053915" "GSM1053922" "GSM1053929" "GSM1053926" "GSM1053925"
 [6] "GSM1053933" "GSM1053937" "GSM1053918" "GSM1053917" "GSM1053935"
[11] "GSM1053919" "GSM1053924" "GSM1053938" "GSM1053941" "GSM1053942"
[16] "GSM1053939" "GSM1053940"


## Stage 2 QC findings

The included 17 samples (12 DCM, 5 controls) have observed expression values from 1.83259 to 14.1064 on the GEO-reported RMA-normalized log2 scale. The included matrix contains 0 missing values and 0 non-finite values. No features were omitted for missing/non-finite values, and no zero-variance features were found; all 33,297 features were used for PCA, classical MDS, and hierarchical clustering. No imputation was needed.

The expression distributions are broadly similar across samples. **GSM1053915** and **GSM1053918** were flagged as possible outliers because they separate from the main groups in PCA/MDS and form long branches in the sample dendrogram. These are visual flags only; all 17 samples remain included. The limma analysis is documented in Stage 3 below.

In [13]:
# Record versions used for the Stage 2 QC figures and the R environment.
cat("R version:", R.version.string, "\n")
cat("tidyverse:", as.character(packageVersion("tidyverse")), "\n")
cat("ggplot2:", as.character(packageVersion("ggplot2")), "\n")
cat("readr:", as.character(packageVersion("readr")), "\n")
cat("\nSession information:\n")
print(utils::sessionInfo())

R version: R version 4.5.2 (2025-10-31) 


tidyverse: 2.0.0 


ggplot2: 4.0.3 


readr: 2.2.0 



Session information:


R version 4.5.2 (2025-10-31)
Platform: x86_64-pc-linux-gnu
Running under: Ubuntu 26.04 LTS

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/blas/libblas.so.3.12.1 
LAPACK: /usr/lib/x86_64-linux-gnu/lapack/liblapack.so.3.12.1;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

time zone: Etc/UTC
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] lubridate_1.9.5 forcats_1.0.1   stringr_1.6.0   dplyr_1.2.1    
 [5] purrr_1.2.2     readr_2.2.0     tidyr_1.3.2     tibble_3.3.1   
 [9] ggplot2_4.0.3   tidyverse_2.0.0

loaded via a namespace (and not attached):
 [1] bit_4.6.0          gtable_0.3.6     

In [ ]:
## Stage 3: GSE42955 differential expression

Use all 17 previously approved samples. Check GPL6244 annotation against the processed matrix feature IDs, remove platform control IDs if represented, and fit a disease-status-only limma model. The Control reference makes the assigned_groupDCM coefficient the DCM-minus-control log2 fold change. BH adjustment is across tested biological features; significance is adjusted $p < 0.05$. The selected gene-symbol table keeps one unambiguous mapped symbol per gene, choosing smallest adjusted p-value, then larger absolute log2FC, then lexically first probe ID. No consensus analysis is run in this stage.

In [17]:
# Load limma for moderated linear models and tidyverse for table operations.
suppressPackageStartupMessages(library(tidyverse))
suppressPackageStartupMessages(library(limma))

# Keep the approved cohort and set Control first so the model coefficient is DCM minus Control.
analysis_manifest <- sample_manifest |>
  dplyr::filter(inclusion_status == "included") |>
  dplyr::mutate(assigned_group = factor(assigned_group, levels = c("Control", "DCM"))) |>
  dplyr::arrange(assigned_group, gsm)
stopifnot(nrow(analysis_manifest) == 17L, dplyr::n_distinct(analysis_manifest$gsm) == 17L)
included_gsms <- analysis_manifest$gsm

# Align expression columns to the manifest before fitting the model.
analysis_expression_table <- expression_table |>
  dplyr::select(probe_id = 1, dplyr::all_of(included_gsms))
analysis_expression <- analysis_expression_table |>
  dplyr::select(-probe_id) |>
  as.matrix()
storage.mode(analysis_expression) <- "double"
rownames(analysis_expression) <- as.character(analysis_expression_table$probe_id)
stopifnot(all(is.finite(analysis_expression)))

# Download the GPL6244 platform annotation once so feature filtering is traceable.
annotation_dir <- file.path(project_root, "data", "processed")
dir.create(annotation_dir, recursive = TRUE, showWarnings = FALSE)
annotation_file <- file.path(annotation_dir, "GPL6244.annot.gz")
annotation_url <- "https://ftp.ncbi.nlm.nih.gov/geo/platforms/GPL6nnn/GPL6244/annot/GPL6244.annot.gz"
if (!file.exists(annotation_file)) {
  download.file(annotation_url, destfile = annotation_file, mode = "wb", method = "libcurl")
}
stopifnot(file.exists(annotation_file), file.info(annotation_file)$size > 0)

# Parse only the GEO platform feature table and record its stated annotation date.
annotation_lines <- readr::read_lines(annotation_file, progress = FALSE)
platform_marker <- match("!platform_table_begin", annotation_lines)
platform_end <- match("!platform_table_end", annotation_lines)
annotation_date_line <- annotation_lines[stringr::str_detect(annotation_lines, "^!Annotation_date")]
platform_line <- annotation_lines[stringr::str_detect(annotation_lines, "^!Annotation_platform =")]
stopifnot(
  !is.na(platform_marker), !is.na(platform_end), length(annotation_date_line) == 1L,
  length(platform_line) == 1L, stringr::str_detect(platform_line, "GPL6244")
)
platform_annotation_date <- sub("^!Annotation_date = ", "", annotation_date_line)
annotation_table <- readr::read_tsv(
  annotation_file, skip = platform_marker,
  n_max = platform_end - platform_marker - 2L,
  show_col_types = FALSE, progress = FALSE
) |>
  dplyr::rename(probe_id = ID) |>
  dplyr::mutate(probe_id = as.character(probe_id)) |>
  dplyr::rename_with(~ "gene_symbol", dplyr::any_of("Gene symbol"))
stopifnot(
  "gene_symbol" %in% names(annotation_table),
  nrow(annotation_table) == nrow(analysis_expression),
  dplyr::n_distinct(annotation_table$probe_id) == nrow(annotation_table),
  setequal(annotation_table$probe_id, rownames(analysis_expression))
)

# The GPL table has no control-class field; screen feature IDs for standard Affymetrix controls.
control_id_pattern <- "^(AFFX|CONTROL|POSITIVE|NEGATIVE|BLANK)"
control_probe_ids <- annotation_table |>
  dplyr::filter(stringr::str_detect(probe_id, stringr::regex(control_id_pattern, ignore_case = TRUE))) |>
  dplyr::pull(probe_id)
biological_probe_ids <- setdiff(rownames(analysis_expression), control_probe_ids)
analysis_expression <- analysis_expression[biological_probe_ids, , drop = FALSE]
stopifnot(length(biological_probe_ids) > 0L)

# Fit one disease-status coefficient; the ordered factor sets non-failing controls as reference.
design <- stats::model.matrix(~ assigned_group, data = analysis_manifest)
colnames(design) <- make.names(colnames(design))
print(colnames(design))
fit <- limma::lmFit(analysis_expression, design)
fit <- limma::eBayes(fit)
probe_results <- limma::topTable(
  fit, coef = "assigned_groupDCM", number = Inf,
  adjust.method = "BH", sort.by = "P"
) |>
  tibble::rownames_to_column("probe_id")

# Add group means for an interpretable check of the estimated DCM-minus-control effect.
control_gsms <- analysis_manifest |>
  dplyr::filter(assigned_group == "Control") |>
  dplyr::pull(gsm)
dcm_gsms <- analysis_manifest |>
  dplyr::filter(assigned_group == "DCM") |>
  dplyr::pull(gsm)
group_means <- tibble::tibble(
  probe_id = rownames(analysis_expression),
  mean_control = rowMeans(analysis_expression[, control_gsms, drop = FALSE]),
  mean_dcm = rowMeans(analysis_expression[, dcm_gsms, drop = FALSE])
)
probe_results <- probe_results |>
  dplyr::left_join(group_means, by = "probe_id") |>
  dplyr::left_join(annotation_table, by = "probe_id") |>
  dplyr::arrange(adj.P.Val, dplyr::desc(abs(logFC)), probe_id)

# Keep the complete probe table, including mapped, ambiguous, and unmapped features.
table_dir <- file.path(project_root, "results", "tables")
dir.create(table_dir, recursive = TRUE, showWarnings = FALSE)
probe_file <- file.path(table_dir, "GSE42955_limma_probe_level_all.csv")
readr::write_csv(probe_results, probe_file)

# Select one well-mapped probe per symbol using the approved significance and tie-break rules.
gene_results <- probe_results |>
  dplyr::mutate(gene_symbol = stringr::str_trim(gene_symbol)) |>
  dplyr::filter(
    !is.na(gene_symbol), !gene_symbol %in% c("", "---", "NA"),
    !stringr::str_detect(gene_symbol, "///|;")
  ) |>
  dplyr::arrange(adj.P.Val, dplyr::desc(abs(logFC)), probe_id) |>
  dplyr::distinct(gene_symbol, .keep_all = TRUE) |>
  dplyr::arrange(adj.P.Val, dplyr::desc(abs(logFC)), probe_id)
gene_file <- file.path(table_dir, "GSE42955_limma_gene_symbol_selected.csv")
readr::write_csv(gene_results, gene_file)

# Save significant probe and gene-symbol subsets using the approved BH threshold.
significant_probes <- probe_results |>
  dplyr::filter(adj.P.Val < 0.05) |>
  dplyr::arrange(adj.P.Val, dplyr::desc(abs(logFC)), probe_id)
significant_genes <- gene_results |>
  dplyr::filter(adj.P.Val < 0.05) |>
  dplyr::arrange(adj.P.Val, dplyr::desc(abs(logFC)), probe_id)
significant_probe_file <- file.path(table_dir, "GSE42955_limma_probe_level_significant.csv")
significant_gene_file <- file.path(table_dir, "GSE42955_limma_gene_symbol_significant.csv")
readr::write_csv(significant_probes, significant_probe_file)
readr::write_csv(significant_genes, significant_gene_file)

# Summarize the tested features and display the top probe's means and estimated effect.
significant_probe_count <- nrow(significant_probes)
large_effect_significant_count <- significant_probes |>
  dplyr::filter(abs(logFC) >= 1) |>
  nrow()
top_probe <- probe_results |>
  dplyr::slice_head(n = 1) |>
  dplyr::select(probe_id, mean_control, mean_dcm, logFC, P.Value, adj.P.Val, gene_symbol)
cat("Included samples:", nrow(analysis_manifest), "(controls", length(control_gsms), "; DCM", length(dcm_gsms), ")\n")
cat("GPL6244 annotation date:", platform_annotation_date, "\n")
cat("Platform annotation has a control-class field:", any(stringr::str_detect(names(annotation_table), stringr::regex("control|probe.type", ignore_case = TRUE))), "\n")
cat("Control-like platform IDs removed:", length(control_probe_ids), "\n")
cat("Biological probes tested:", nrow(analysis_expression), "\n")
cat("Significant probes (BH adjusted p < 0.05):", significant_probe_count, "\n")
cat("Significant probes with |log2FC| >= 1:", large_effect_significant_count, "\n")
cat("Selected gene symbols:", nrow(gene_results), "\n")
cat("Significant selected gene symbols:", nrow(significant_genes), "\n")
cat("Top-ranked probe:\n")
print(top_probe)
cat("Saved tables:\n")
print(c(probe_file, gene_file, significant_probe_file, significant_gene_file))
cat("Reference-setting code: factor(assigned_group, levels = c(\"Control\", \"DCM\")); model.matrix(~ assigned_group, data = analysis_manifest)\n")
cat("\nlimma version:", as.character(packageVersion("limma")), "\n")
cat("\nSession information:\n")
print(utils::sessionInfo())

[1] "X.Intercept."      "assigned_groupDCM"


Included samples: 17 (controls 5 ; DCM 12 )


GPL6244 annotation date: Aug 09 2016 


Platform annotation has a control-class field: FALSE 


Control-like platform IDs removed: 0 


Biological probes tested: 33297 


... (8 outputs left out)

Reference-setting code: factor(assigned_group, levels = c("Control", "DCM")); model.matrix(~ assigned_group, data = analysis_manifest)



limma version: 3.66.0 



Session information:


R version 4.5.2 (2025-10-31)
Platform: x86_64-pc-linux-gnu
Running under: Ubuntu 26.04 LTS

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/blas/libblas.so.3.12.1 
LAPACK: /usr/lib/x86_64-linux-gnu/lapack/liblapack.so.3.12.1;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

time zone: Etc/UTC
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] limma_3.66.0    lubridate_1.9.5 forcats_1.0.1   stringr_1.6.0  
 [5] dplyr_1.2.1     purrr_1.2.2     readr_2.2.0     tidyr_1.3.2    
 [9] tibble_3.3.1    ggplot2_4.0.3   tidyverse_2.0.0

loaded via a namespace (and not attached):
 [1] bit_4.6.0          g

## Stage 3 results

All 17 included samples were retained (5 controls and 12 DCM). GPL6244 annotation dated Aug 09 2016 was used. The annotation did not provide an explicit control-probe class field, so the documented Affymetrix control-ID prefix screen identified 0 controls to remove; 33,297 biological probes were tested.

The limma model used disease status only, with BH adjustment. There were 8 probes with adjusted p < 0.05; 4 of these had absolute log2FC >= 1. The gene-symbol selected table contains 19,036 symbols, of which 6 are significant. Ambiguous or unmapped probes remain in probe-level results and are excluded from symbol selection.

Top-ranked probe: 7897378 (PER3). Mean control expression = 7.069542; mean DCM expression = 6.009144; log2FC (DCM minus control) = -1.060398; adjusted p = 0.001879854.

Reference-setting code: factor(assigned_group, levels = c("Control", "DCM")); model.matrix(~ assigned_group, data = analysis_manifest).

Saved tables: GSE42955_limma_probe_level_all.csv, GSE42955_limma_gene_symbol_selected.csv, GSE42955_limma_probe_level_significant.csv, and GSE42955_limma_gene_symbol_significant.csv. This completes Stage 3; consensus analysis has not been run.
